# INSTRUCTOR SOLUTION: Exploration Strategies
## AIAT 123 - Reinforcement Learning

**⚠️ INSTRUCTOR USE ONLY**

Complete solution for multi-armed bandit and exploration strategies.

## Inputs and Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [ ]:
%pip install numpy matplotlib -q
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
print('✅ Setup complete!')


## Solution: Multi-Armed Bandit

Simulate ads with different click-through rates

In [ ]:
class MultiArmedBandit:
    def __init__(self, n_arms, true_rewards):
        self.n_arms = n_arms
        self.true_rewards = np.asarray(true_rewards, dtype=float)

    def pull(self, arm: int) -> float:
        p = self.true_rewards[arm]
        return float(np.random.binomial(1, p))

## Solution: Epsilon-Greedy

In [ ]:
def epsilon_greedy(bandit, epsilon=0.1, n_steps=1000):
    q = np.zeros(bandit.n_arms)
    n = np.zeros(bandit.n_arms)
    rewards = []
    for _ in range(n_steps):
        if np.random.random() < epsilon:
            a = np.random.randint(0, bandit.n_arms)
        else:
            a = int(np.argmax(q))
        r = bandit.pull(a)
        n[a] += 1
        q[a] += (r - q[a]) / n[a]
        rewards.append(r)
    return q, np.array(rewards)


## Solution: UCB Algorithm

In [ ]:
def ucb(bandit, c=2, n_steps=1000):
    n = np.zeros(bandit.n_arms)
    q = np.zeros(bandit.n_arms)
    rewards = []
    for t in range(1, n_steps + 1):
        unc = np.where(n == 0, np.inf, c * np.sqrt(np.log(t) / n))
        a = int(np.argmax(q + unc))
        r = bandit.pull(a)
        n[a] += 1
        q[a] += (r - q[a]) / n[a]
        rewards.append(r)
    return q, np.array(rewards)


## Solution: Comparison

In [ ]:
def cumulative_regret(bandit, rewards_history):
    """Regret vs always picking the best arm (oracle)."""
    best = float(np.max(bandit.true_rewards))
    inst = best - np.asarray(rewards_history, dtype=float)
    return np.cumsum(inst)


bandit_demo = MultiArmedBandit(5, [0.1, 0.3, 0.5, 0.2, 0.4])
_, r_eg = epsilon_greedy(bandit_demo, epsilon=0.1, n_steps=500)
bandit_demo2 = MultiArmedBandit(5, [0.1, 0.3, 0.5, 0.2, 0.4])
_, r_ucb = ucb(bandit_demo2, c=2.0, n_steps=500)
print('epsilon-greedy mean reward', r_eg.mean())
print('ucb mean reward', r_ucb.mean())
print('✅ Demo complete')


## Teaching Notes

**Key Insights:**
1. UCB typically outperforms epsilon-greedy
2. UCB balances exploration/exploitation automatically
3. Epsilon-greedy needs tuning of epsilon parameter
4. Real-world: A/B testing, ad optimization, recommendation

**Grading: 100 points**
- Bandit implementation: 20 pts
- Epsilon-greedy: 25 pts
- UCB: 30 pts
- Comparison: 25 pts

**Common Mistakes:**
- Not initializing UCB properly
- Wrong UCB formula
- Not tracking optimal actions
- Poor visualization

**Real-World Applications:**
- Google Ads: Ad selection
- Netflix: Content recommendation
- Amazon: Product placement
- Clinical trials: Treatment selection